In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [ ]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

# DB-Specific

In [ ]:
from lib import traxsource
mio   = traxsource.MusicDBIO(verbose=False)
webio = traxsource.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

# Master Files

In [ ]:
if False:
    starter = io.get("starter.p")
    starter = starter[starter["Name"].str.len() > 0]
    starter.index = starter["URL"].map(mio.getdbid)    
    mio.data.saveSearchArtistData(data=starter)

In [ ]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
localXtraArtists   = MusicDBData(path=permDir, fname="{0}SearchedForLocalXtraArtists".format(db.lower()))
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [ ]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Artists:     {0}".format(len(localArtists.get())))
print("   Local XtraArtists: {0}".format(len(localXtraArtists.get())))
print("   Master Artists:    {0}".format(len(masterArtists.get())))
print("   Errors:            {0}".format(len(errors.get())))
print("   Search Artists:    {0}".format(searchArtists.shape[0]))
print("   Known IDs:         {0}".format(len(knownArtists)))

# Download Artist Data

In [ ]:
mio   = traxsource.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = traxsource.RawWebData(debug=False)

In [ ]:
useSearchData = True

if useSearchData is True:
    artistNames      = searchArtists #.sort_values(by="Num", ascending=False)
    localArtistsDict = localArtists.get()
    artistNamesToGet = artistNames[~artistNames.index.isin(localArtistsDict.keys())].sample(frac=1)

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(localArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  20571
#   Artist Names To Get:  13671

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("today", "7:00pm")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["Name"]
    artistURL  = row["Ref"]
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistURL=artistURL)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(5.0)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

In [ ]:
del searchedForErrors['296602']
errors.save(data=searchedForErrors)

In [ ]:
from lib.traxsource import moveLocalFiles
moveLocalFiles()

# Download Extra Artist Files

In [ ]:
mio   = traxsource.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = traxsource.RawWebData(debug=False)

In [ ]:
artistNames = None
for modVal in range(100):
    modValData = mio.data.getModValData(modVal)
    modValPagesData = modValData.apply(lambda rData: (rData.pages.tot, rData.artist.name, rData.url.url)).apply(Series)    
    artistNames = concat([artistNames, modValPagesData[modValPagesData[0] > 1]]) if isinstance(modValPagesData,DataFrame) else  modValPagesData[modValPagesData[0] > 1]
    if (modVal+1) % 10 == 0:
        print(modVal+1,'\t',artistNames.shape[0])
artistNames.columns = ["Pages", "Name", "URL"]
artistNames["URL"] = artistNames["URL"].apply(lambda url: url[26:])
artistNames = artistNames.sort_values("Pages", ascending=False)

localXtraArtistsDict = localXtraArtists.get()
artistNamesToGet = artistNames[~artistNames.index.isin(localXtraArtistsDict.keys())]
print("{0} Search Results".format(db))
print("   Available Names:      {0}".format(len(artistNames)))
print("   Known Artist Names:   {0}".format(len(localXtraArtistsDict)))
print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  1457
#   Artist Names To Get:  4665
#   Artist Names To Get:  3890
#   Artist Names To Get:  1040

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "6:50am")
tt = TermTime("today", "7:00pm")
#tt = TermTime("today", "11:06am")
maxN = 50000000

n  = 0
localXtraArtistsDict = localXtraArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName  = row["Name"]
    artistURL   = row["URL"]
    artistPages = row["Pages"]
    if localXtraArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    good = True
    for page in range(2,artistPages+1):
        response = webio.getArtistData(artistName=artistName, artistURL=artistURL, page=page)
        if response is None or len(response) == 0:
            print("Error ==> {0}".format((artistID,artistName)))
            searchedForErrors[artistID] = artistName
            errors.save(data=searchedForErrors)
            webio.sleep(5.0)
            good = False
            break

        mio.data.saveRawArtistExtraData(data=response, modval=mio.getModVal(artistID), dbID="{0}-{1}".format(artistID,page))
        webio.sleep(5.0)

    if good is True:
        localXtraArtistsDict[artistID] = artistName        
        n += 1
        
    if n % 5 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localXtraArtistsDict), db))
        localXtraArtists.save(data=localXtraArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localXtraArtistsDict), db))
localXtraArtists.save(data=localXtraArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

# Create Starter

In [ ]:
from collections import Counter
from ioutils import FileIO, HTMLIO
from timeutils import Timestat
io = FileIO()
hio = HTMLIO()
cntr = Counter()
refs = []
mio   = traxsource.MusicDBIO()
ts = Timestat("Creating Starter")
for modVal in range(100):
    for ifile in mio.dir.getRawModValDataDir(modVal).glob("*.p", debug=False):
        bsdata = hio.get(io.get(ifile))
        refs  += [{"Name": ref.text, "URL": ref.get('href')} for ref in bsdata.findAll("a") if ref.attrs['href'].startswith("/artist/")]
    ts.update(n=modVal+1, N=100)
    if (modVal+1) % 5 == 0:
        print(modVal+1,'\t',len(refs))
ts.stop()    

In [ ]:
df = DataFrame(refs)
df.index = df["URL"].map(mio.getdbid)
N  = df.index.value_counts()
N.name = "Num"
df = df.drop_duplicates(subset=['URL'])
df.index.name = None
df.columns = ["Name", "Ref"]
df = df.join(N)
df = df.sort_values(by='Num', ascending=False)
mio.data.saveSearchArtistData(data=df)

# Move Local

In [ ]:
from lib.traxsource import moveLocalFiles

In [ ]:
moveLocalFiles(verbose=True)

In [ ]:
from utils import PoolIO
pio = PoolIO("Traxsource")
pio.parse()
pio.merge()
pio.meta()
pio.sum()
pio.search()